In [9]:
import duckdb
import pandas as pd
db_path = "./data/staging/landed.duckdb"
# %%
with duckdb.connect(str(db_path), read_only=True) as conn:
    tables = conn.execute("SELECT table_name FROM duckdb_tables();").fetchall()

    data = []
    for (table_name,) in tables:
        count = conn.execute(f'SELECT count(*) FROM "{table_name}"').fetchone()[0]
        data.append({"table_name": table_name, "row_count": count})

    df_summary = (
        pd.DataFrame(data)
        .sort_values(by="row_count", ascending=False)
        .reset_index(drop=True)
    )

display(df_summary)

,table_name,row_count
0,aemo_nem_dispatch,1358005
1,openelectricity_mix,1270957
2,bom_observations,237570
3,aemo_wem_dispatch,211104
4,aemo_holidays,390


In [10]:
from pathlib import Path
import pandas as pd
import duckdb



_REGIONS = ["NSW1", "QLD1", "SA1", "TAS1", "VIC1", "WEM"]
_BOM_STATION_MAP = {
    '009225': 'WEM',
    '023034': 'SA1',
    '040913': 'QLD1',
    '066037': 'NSW1',
    '086282': 'VIC1',
    '094029': 'TAS1',
}

def build_master_panel(db_path: str | Path) -> pd.DataFrame:
    """Builds the local, wide 30-minute-grain master training table using Pandas."""
    conn = duckdb.connect(str(db_path), read_only=True)

    # 1. Load Staging Tables from DuckDB into Pandas
    nem_df = conn.execute("SELECT * FROM aemo_nem_dispatch").fetchdf()
    wem_df = conn.execute("SELECT * FROM aemo_wem_dispatch").fetchdf()
    oe_df = conn.execute("SELECT * FROM openelectricity_mix").fetchdf()
    bom_df = conn.execute("SELECT * FROM bom_observations").fetchdf()
    holidays_df = conn.execute("SELECT DISTINCT date, region, holiday_name, is_workday FROM aemo_holidays").fetchdf()

    conn.close()

    # Ensure timestamp dtypes
    for df in [nem_df, wem_df, oe_df, bom_df]:
        if 'ts' in df.columns:
            df['ts'] = pd.to_datetime(df['ts'])

    # 2. Process AEMO Demand & Dispatch (Union NEM + WEM with column alignment)
    nem_df['diesel_mw'] = None
    nem_df['biomass_mw'] = None
    nem_df['total_generation_mw'] = None

    wem_df['hydro_mw'] = None
    wem_df['net_import_mw'] = None

    aemo_combined = pd.concat([nem_df, wem_df], ignore_index=True)
    aemo_combined['bucket_ts'] = aemo_combined['ts'].dt.floor('30min')

    aemo_agg_cols = {
        'demand_mw': 'aemo_demand_mw',
        'price_mwh': 'aemo_price_mwh',
        'coal_mw': 'aemo_coal_mw',
        'gas_mw': 'aemo_gas_mw',
        'hydro_mw': 'aemo_hydro_mw',
        'wind_mw': 'aemo_wind_mw',
        'solar_utility_mw': 'aemo_solar_utility_mw',
        'solar_rooftop_mw': 'aemo_solar_rooftop_mw',
        'battery_mw': 'aemo_battery_mw',
        'net_import_mw': 'aemo_net_import_mw',
        'diesel_mw': 'aemo_diesel_mw',
        'biomass_mw': 'aemo_biomass_mw',
        'total_generation_mw': 'aemo_total_generation_mw'
    }

    aemo_demand = (
        aemo_combined.groupby(['bucket_ts', 'region'])[list(aemo_agg_cols.keys())]
        .mean()
        .rename(columns=aemo_agg_cols)
        .reset_index()
    )

    # 3. Process OpenElectricity Mix
    oe_df['bucket_ts'] = oe_df['ts'].dt.floor('30min')
    oe_agg_cols = {
        'demand_mw': 'oe_demand_mw',
        'price_mwh': 'oe_price_mwh',
        'intensity_kg_per_mwh': 'oe_intensity_kg_per_mwh',
        'battery_discharge_mw': 'oe_battery_discharge_mw',
        'battery_charge_mw': 'oe_battery_charge_mw',
        'biomass_mw': 'oe_biomass_mw',
        'coal_mw': 'oe_coal_mw',
        'distillate_mw': 'oe_distillate_mw',
        'gas_mw': 'oe_gas_mw',
        'hydro_mw': 'oe_hydro_mw',
        'pumped_hydro_mw': 'oe_pumped_hydro_mw',
        'solar_rooftop_mw': 'oe_solar_rooftop_mw',
        'solar_utility_mw': 'oe_solar_utility_mw',
        'wind_mw': 'oe_wind_mw',
        'total_generation_mw': 'oe_total_generation_mw',
        'total_renewable_mw': 'oe_total_renewable_mw'
    }

    oe_mix = (
        oe_df.groupby(['bucket_ts', 'region'])[list(oe_agg_cols.keys())]
        .mean()
        .rename(columns=oe_agg_cols)
        .reset_index()
    )

    # 4. Process Weather Observations (Map stations to regions and bucket timestamps)
    bom_df['region'] = bom_df['station_id'].map(_BOM_STATION_MAP)
    bom_df = bom_df.dropna(subset=['region'])
    bom_df['bucket_ts'] = bom_df['ts'].dt.floor('30min')

    weather_agg_cols = {
        'temp_c': 'temp_c',
        'apparent_temp_c': 'apparent_temp_c',
        'dew_point_c': 'dew_point_c',
        'humidity_pct': 'humidity_pct',
        'wind_speed_kmh': 'wind_speed_kmh',
        'wind_direction_deg': 'wind_direction_deg',
        'wind_gust_kmh': 'wind_gust_kmh',
        'pressure_hpa': 'pressure_hpa',
        'rain_since_9am_mm': 'rain_since_9am_mm',
        'cloud_oktas': 'cloud_oktas'
    }

    weather = (
        bom_df.groupby(['bucket_ts', 'region'])[list(weather_agg_cols.keys())]
        .mean()
        .rename(columns=weather_agg_cols)
        .reset_index()
        .sort_values('bucket_ts')
    )

    # 5. Construct Full Rectangular Grid (bucket_ts × region)
    all_buckets = pd.concat([aemo_demand['bucket_ts'], oe_mix['bucket_ts']]).unique()
    grid_index = pd.MultiIndex.from_product([all_buckets, _REGIONS], names=['bucket_ts', 'region'])
    grid = pd.DataFrame(index=grid_index).reset_index()

    # 6. Merge Datasets onto the Grid
    master = grid.merge(aemo_demand, on=['bucket_ts', 'region'], how='left')
    master = master.merge(oe_mix, on=['bucket_ts', 'region'], how='left')

    # Pandas ASOF Merge Equivalent (requires sorted data frames by the 'on' column and matching 'by' column)
    master = master.sort_values('bucket_ts')
    weather = weather.sort_values('bucket_ts')

    master = pd.merge_asof(
        master,
        weather,
        on='bucket_ts',
        by='region',
        direction='backward'
    )

    # 7. Merge Holidays
    holidays_df['date'] = pd.to_datetime(holidays_df['date']).dt.date
    master['date_key'] = master['bucket_ts'].dt.date

    master = master.merge(
        holidays_df,
        left_on=['date_key', 'region'],
        right_on=['date', 'region'],
        how='left'
    )
    master.drop(columns=['date_key', 'date'], inplace=True, errors='ignore')
    master['is_holiday'] = master['holiday_name'].notna()

    # Final Sort Order
    master = master.sort_values(['bucket_ts', 'region']).reset_index(drop=True)
    return master

df = build_master_panel(db_path)